# LlamaIndex와 AgentCore 메모리 - 투자 포트폴리오 어드바이저 (장기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 **장기 메모리**를 활용한 투자 포트폴리오 어드바이저를 만드는 방법을 보여줍니다. 여러 고객 미팅과 시장 사이클에 걸쳐 지속적으로 투자 지식을 축적하고, 수개월에서 수년에 걸친 포트폴리오 성과를 추적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: 장기 크로스 세션 메모리
- **에이전트 사용 사례**: 투자 포트폴리오 어드바이저
- **에이전트 프레임워크**: LlamaIndex
- **LLM 모델**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex 에이전트, 금융 도구
- **예제 복잡도**: 고급

## 비즈니스 가치

**기업 투자 인텔리전스**: 포트폴리오 지식을 축적하고, 투자 진화를 추적하며, 분기와 연도에 걸쳐 포괄적인 시장 분석을 유지하는 지속적 AI 메모리로 자산 관리 업무를 혁신하세요.

**주요 전문적 이점:**
- **포트폴리오 연속성**: 투자 기간 및 팀원 간 원활한 지식 이전
- **투자 기억**: 중요한 시장 인사이트, 전략 및 성과 데이터를 영구적으로 보존
- **크로스 포트폴리오 인텔리전스**: 여러 고객 포트폴리오에 걸친 패턴과 연관성 파악
- **전략적 우수성**: 과거 성과 데이터를 활용한 우수한 투자 의사결정
- **고객 관계**: 다년간 자산 관리를 위한 상세한 컨텍스트 유지
- **리스크 관리**: 시장 사이클과 투자 전략에 미치는 영향 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼은 12개월 보존을 위한 시맨틱 전략과 함께 AgentCore 메모리를 사용합니다:
- **메모리 유형**: 자동 인사이트 추출을 위한 시맨틱 전략
- **보존 기간**: 포트폴리오 연속성을 위한 365일 이벤트 만료
- **크로스 세션**: 동일 actor_id + memory_id, 투자 기간별 다른 session_id
- **검색 기능**: 포트폴리오 이력 전반에 걸친 시맨틱 검색을 위한 내장 메모리 검색 도구

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **시맨틱 전략 구성**: 365일 보존의 자동 인사이트 추출을 위한 SemanticStrategy 사용
2. **크로스 세션 지속성**: 동일 actor_id + memory_id, 기간별 다른 session_id로 지식 연속성 확보
3. **커스텀 메모리 검색 도구**: AgentCore의 네이티브 search_long_term_memories()를 LlamaIndex FunctionTool로 래핑
4. **시맨틱 처리 파이프라인**: 대화 이벤트 → 시맨틱 메모리 변환을 위한 90초 대기
5. **동적 세션 관리**: 유연한 세션 처리를 위한 memory.context.session_id 사용

**학습 내용:**

- 여러 고객 미팅에 걸친 지속적 AgentCore 메모리 생성
- 시간에 따른 누적 투자 지식 구축
- 시장 조사 및 고객 이력 전반에 걸친 시맨틱 검색 구현
- 포트폴리오 진화 및 투자 성과 추적
- 크로스 세션 금융 지식 지속성 및 검색 테스트

## 시나리오 컨텍스트

이 예제에서는 분기와 연도에 걸친 여러 미팅에서 고객 투자 지식을 유지하는 "투자 포트폴리오 어드바이저"를 만듭니다. 어드바이저는 AgentCore 메모리를 사용하여 고객 프로필, 포트폴리오 성과, 시장 인사이트, 투자 결과에 대한 지속적 지식 베이스를 구축하며, 이는 시간이 지남에 따라 성장하고 진화하여 정교한 종합적 자산 관리를 가능하게 합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 종속성 설치 및 설정

In [ ]:
# 시맨틱 전략 툴킷을 포함한 필수 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필수 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import SemanticStrategy
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("모든 종속성을 성공적으로 임포트했습니다!")

## 2단계: AgentCore 메모리 구성

장기 투자 지식을 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# 장기 지속성을 위한 시맨틱 전략으로 AgentCore 메모리 생성
region = os.getenv('AWS_REGION', 'us-east-1')
memory_manager = MemoryManager(region_name=region)

try:
    # 자동 인사이트 추출을 위한 시맨틱 전략으로 메모리 생성
    memory = memory_manager.get_or_create_memory(
        name=f'InvestmentAdvisorSemantic_{int(datetime.now().timestamp())}',
        strategies=[SemanticStrategy(name="investmentLongTermMemory")],
        event_expiry_days=365  # 투자 기록을 위한 12개월 보존
    )
    memory_id = memory.get('id')
    print(f"시맨틱 메모리 생성 완료: {memory_id}")
    print(f"   상태: {memory.get('status')}")
    print(f"   전략: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")
    
    # 메모리가 ACTIVE 상태가 될 때까지 대기
    if memory.get('status') != 'ACTIVE':
        print(f"\n메모리가 ACTIVE 상태가 될 때까지 대기 중 (현재 {memory.get('status')})...")
        import time
        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get('status')
            print(f"   [{waited}초] 상태: {status}")
            if status == 'ACTIVE':
                print(f"메모리가 ACTIVE 상태가 되었습니다! ({waited}초 소요)")
                break
        else:
            print(f"경고: {max_wait}초 후에도 메모리가 ACTIVE 상태가 아닙니다. 계속 진행합니다...")
    
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 투자 도구 구현

종합적 자산 관리를 위한 전문 도구를 정의합니다:

In [ ]:
def record_client_meeting(client_id: str, meeting_type: str, portfolio_value: str, key_decisions: str) -> str:
    """Record client meeting with portfolio updates and decisions"""
    return f"고객 미팅 기록 완료: {client_id}의 {meeting_type} (${portfolio_value})"

def track_portfolio_performance(client_id: str, period: str, return_pct: str, benchmark_return: str, attribution: str) -> str:
    """Track portfolio performance vs benchmark with attribution analysis"""
    return f"포트폴리오 성과 기록: {client_id} {period}: {return_pct} vs {benchmark_return}"

def document_market_insight(insight_type: str, market_event: str, impact_assessment: str, client_implications: str) -> str:
    """Document market insight with client portfolio implications"""
    print(f"시장 인사이트: {insight_type} - {market_event} (영향: {impact_assessment})")
    return f"시장 인사이트 기록 완료: {insight_type}"

def update_investment_thesis(client_id: str, asset_class: str, thesis: str, conviction_level: str) -> str:
    """Update investment thesis for specific asset class"""
    print(f"투자 논제: {client_id} - {asset_class} ({conviction_level} 확신도)")
    return f"투자 논제 업데이트 완료: {client_id}"

def log_rebalancing_action(client_id: str, action_type: str, securities: str, rationale: str) -> str:
    """Log portfolio rebalancing actions with rationale"""
    print(f"리밸런싱: {client_id} - {action_type}: {securities}")
    return f"리밸런싱 기록 완료: {client_id}"

def log_advisory_milestone(quarter: str, milestone: str, details: str) -> str:
    """Log an advisory milestone with quarter and detailed progress"""
    print(f"자문 마일스톤 - {quarter}: {milestone}")
    return f"{quarter} 마일스톤 기록 완료: {milestone} - {details}"

def track_investment_metrics(metric_type: str, value: str, client_id: str, quarter: str) -> str:
    """Track specific investment metrics with client and timeline"""
    print(f"{quarter}: {metric_type} = {value} ({client_id} 대상)")
    return f"{metric_type} 추적 완료: {client_id}의 {quarter} - {value}"

def save_advisory_insight(insight: str, quarter: str, market_context: str) -> str:
    """Save advisory insights with market context"""
    print(f"{quarter} 인사이트: {insight[:50]}...")
    return f"{quarter} 인사이트 저장 완료 (시장 컨텍스트: {market_context})"

# 에이전트를 위한 도구 객체 생성
investment_tools = [
    FunctionTool.from_defaults(fn=record_client_meeting),
    FunctionTool.from_defaults(fn=track_portfolio_performance),
    FunctionTool.from_defaults(fn=document_market_insight),
    FunctionTool.from_defaults(fn=update_investment_thesis),
    FunctionTool.from_defaults(fn=log_rebalancing_action),
    FunctionTool.from_defaults(fn=log_advisory_milestone),
    FunctionTool.from_defaults(fn=track_investment_metrics),
    FunctionTool.from_defaults(fn=save_advisory_insight)
]

print("투자 도구 생성 완료!")

## 3b단계: 메모리 검색 도구 추가

에이전트가 장기 메모리를 검색할 수 있는 도구를 생성합니다:

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """Create a tool for the agent to search its own long-term memory"""
    
    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant information about clients, portfolios, past decisions, and market insights.
        
        Use this tool when you need to recall:
        - Client information (portfolio values, risk profiles, investment goals)
        - Past investment decisions and their outcomes
        - Portfolio performance history
        - Market insights and their applications
        - Investment theses and their evolution
        
        Args:
            query: Search query describing what information you need (e.g., 'CLIENT-001 portfolio', 'investment theses', 'Q1 performance')
        
        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager
            
            # 세션 매니저 생성 (memory_id와 region만 필요)
            session_manager = MemorySessionManager(
                memory_id=memory_id,
                region_name=region
            )
            
            # 시맨틱 전략 네임스페이스에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # 시맨틱 전략 네임스페이스에서 검색
                top_k=5,
                max_results=10
            )
            
            if not results:
                return "장기 메모리에서 관련 정보를 찾지 못했습니다. 새로운 정보이거나 메모리 추출이 아직 처리 중일 수 있습니다."
            
            # 에이전트를 위한 결과 포맷팅
            output = "장기 메모리에서 검색된 정보:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord 객체 - content 속성에 접근
                content = getattr(result, 'content', str(result))
                # 매우 긴 콘텐츠 잘라내기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"
            
            return output
            
        except Exception as e:
            return f"메모리 검색 오류: {str(e)}. 과거 컨텍스트 없이 진행합니다."
    
    return FunctionTool.from_defaults(fn=search_long_term_memory)

# 메모리 검색 도구 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "financial-advisor", region)

# 도구 목록에 메모리 검색 추가
investment_tools_with_memory = investment_tools + [memory_search_tool]

print(f"메모리 검색 도구 생성 완료! 총 도구 수: {len(investment_tools_with_memory)}")
print("   사용 네임스페이스: /strategies/ (시맨틱 전략 호환용)")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: 멀티 세션 에이전트 구현

다양한 자문 기간을 시뮬레이션하기 위한 헬퍼 함수를 생성합니다:

In [ ]:
# 장기 메모리(크로스 세션) 구성
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ADVISOR_ID = "financial-advisor"  # 모든 세션에서 동일한 어드바이저

def create_advisory_session(session_name: str):
    """장기 메모리 지속성을 갖춘 새로운 자문 세션 생성"""
    context = AgentCoreMemoryContext(
        actor_id=ADVISOR_ID,         # 동일한 어드바이저
        memory_id=memory_id,         # 동일한 메모리 저장소 (장기 메모리 활성화)
        session_id=f"advisory-{session_name}", # 기간별 다른 세션
        namespace="/wealth-management/"
    )
    
    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=investment_tools_with_memory,  # 메모리 검색 기능이 포함된 도구 사용
        llm=llm, 
        verbose=True,  # 메모리 검색 시점을 확인하기 위해 verbose 활성화
        system_prompt="""You are a senior investment advisor with access to long-term memory.
        
CRITICAL: When asked about clients, portfolios, past decisions, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What clients am I managing?" → Use search_long_term_memory("clients portfolio")
- "What was CLIENT-001's performance?" → Use search_long_term_memory("CLIENT-001 performance")
- "What investment theses do I have?" → Use search_long_term_memory("investment thesis")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses."""
    )
    
    return agent, memory

print("멀티 세션 투자 포트폴리오 어드바이저 설정 완료!")

## 5단계: Q1 자문 세션 - 초기 고객 온보딩

첫 번째 자문 세션을 시작하고 고객 기준선을 설정합니다:

In [ ]:
# === Q1 자문 세션 ===
print("=== Q1: 초기 고객 온보딩 ===")

agent_q1, memory_q1 = create_advisory_session("q1")

# 초기 고객 미팅 기록
# (시니어 어드바이저 Jennifer Walsh로서, CLIENT-001의 초기 포트폴리오 리뷰 미팅을 기록 요청. 포트폴리오 가치 $3,200,000, 주요 결정사항: 중간-공격적 리스크 프로필 설정, 20년 투자 기간, 목표 배분 주식 70%/채권 25%/대안투자 5%)
response = await agent_q1.run(
    "I'm Senior Advisor Jennifer Walsh. Record client meeting for 'CLIENT-001' with meeting type 'Initial Portfolio Review', "
    "portfolio value '$3,200,000', key decisions 'established moderate-aggressive risk profile, 20-year investment horizon, "
    "target allocation 70% equity/25% fixed income/5% alternatives'.",
    memory=memory_q1
)

print("Q1 초기 미팅:")
print(response)

In [ ]:
# 초기 투자 논제 기록
# (CLIENT-001의 미국 대형주 주식 투자 논제 업데이트: 기술 혁신과 수익 모멘텀으로 성장주 비중확대, 높은 확신도)
response = await agent_q1.run(
    "Update investment thesis for 'CLIENT-001': asset class 'US Large Cap Equity', "
    "thesis 'overweight growth stocks due to technological innovation and earnings momentum', conviction level 'high'.",
    memory=memory_q1
)
print("Q1 주식 논제:", response)

# (CLIENT-001의 채권 투자 논제 업데이트: 금리 상승 환경으로 인한 단기 듀레이션 편향, 신용 품질 중시, 중간 확신도)
response = await agent_q1.run(
    "Update investment thesis for 'CLIENT-001': asset class 'Fixed Income', "
    "thesis 'short duration bias due to rising rate environment, focus on credit quality', conviction level 'medium'.",
    memory=memory_q1
)
print("Q1 채권 논제:", response)

In [ ]:
# 초기 성과 기준 추적
# (CLIENT-001의 포트폴리오 성과 추적: Q1 2024, 수익률 +8.2%, 벤치마크 수익률 +7.1%, 기여도 분석 - 기술주 비중확대 +0.8%, 듀레이션 축소 +0.3%)
response = await agent_q1.run(
    "Track portfolio performance for 'CLIENT-001': period 'Q1 2024', return_pct '+8.2%', "
    "benchmark_return '+7.1%', attribution 'tech overweight +0.8%, duration underweight +0.3%'.",
    memory=memory_q1
)
print("Q1 성과:", response)

# 이벤트가 저장되었는지 확인
print("\nQ1 이벤트 저장 확인 중...")
try:
    client = MemoryClient(region_name=region)
    events = client.list_events(
        memory_id=memory_id,
        actor_id=ADVISOR_ID,
        session_id=memory_q1.context.session_id
    )
    print(f"Q1 세션에 {len(events)}개의 대화 이벤트가 저장되었습니다")
except Exception as e:
    print(f"경고: 이벤트 확인 불가: {e}")

# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n시맨틱 메모리 추출 및 인덱싱 대기 중...")
print("   (AgentCore가 백그라운드에서 대화 이벤트를 처리합니다)")
await asyncio.sleep(90)  # 메모리 추출을 위한 대기 시간 증가 (이전 10초에서 변경)
print("메모리 처리 완료 - 이제 메모리를 검색할 수 있습니다")

## 6단계: Q2 자문 세션 - 시장 변동성 대응

장기 메모리 검색을 테스트하고 시장 변화에 적응합니다:

In [ ]:
# === Q2 자문 세션 ===
print("\n=== Q2: 시장 변동성 대응 (새 세션) ===")

agent_q2, memory_q2 = create_advisory_session("q2")

# 크로스 세션 고객 회상 테스트 - 에이전트가 search_long_term_memory 도구를 사용해야 함
print("\n세션 간 메모리 검색 테스트 중...")
print("   (에이전트가 search_long_term_memory 도구를 사용하는지 확인하세요)\n")

# (관리 중인 고객이 누구인지, 포트폴리오 가치, 리스크 프로필, 투자 논제는 무엇인지 질문)
response = await agent_q2.run(
    "What clients am I managing? What are their portfolio values, risk profiles, and investment theses?",
    memory=memory_q2
)

print("\nQ2 고객 회상:")
print(response)
print("\n예상 결과: CLIENT-001, $3.2M 포트폴리오, 중간-공격적, 성장 주식 논제")

In [ ]:
# 시장 변동성 인사이트 기록
# (시장 인사이트 기록: 유형 '지정학적 리스크', 시장 이벤트 '무역 긴장 고조', 영향 평가 '변동성 증가, 성장주에서 가치주로 섹터 로테이션', 고객 영향 '기술주 비중확대 검토, 방어적 포지셔닝 고려')
response = await agent_q2.run(
    "Document market insight: insight type 'Geopolitical Risk', market event 'Trade tensions escalation', "
    "impact assessment 'increased volatility, sector rotation from growth to value', "
    "client implications 'review tech overweight, consider defensive positioning'.",
    memory=memory_q2
)
print("Q2 시장 인사이트:", response)

# 리밸런싱 대응 기록
# (CLIENT-001의 리밸런싱 조치 기록: 전술적 조정, QQQ 3% 축소, VTV(가치 ETF) 2% 추가, VGSH(단기 국채) 1% 추가, 근거 - 지정학적 불확실성으로 인한 방어적 포지셔닝, 장기 배분 목표 유지)
response = await agent_q2.run(
    "Log rebalancing action for 'CLIENT-001': action type 'Tactical Adjustment', "
    "securities 'reduced QQQ by 3%, increased VTV (value ETF) by 2%, added VGSH (short treasury) by 1%', "
    "rationale 'defensive positioning due to geopolitical uncertainty, maintain long-term allocation targets'.",
    memory=memory_q2
)
print("Q2 리밸런싱:", response)

In [ ]:
# Q2 성과 영향 추적
# (CLIENT-001의 포트폴리오 성과 추적: Q2 2024, 수익률 -2.1%, 벤치마크 수익률 -3.8%, 기여도 분석 - 방어적 포지셔닝 +1.2%, 가치주 편향 +0.5%)
response = await agent_q2.run(
    "Track portfolio performance for 'CLIENT-001': period 'Q2 2024', return_pct '-2.1%', "
    "benchmark_return '-3.8%', attribution 'defensive positioning +1.2%, value tilt +0.5%'.",
    memory=memory_q2
)
print("Q2 성과:", response)

# 성과 비교 테스트
# (CLIENT-001의 Q2 성과가 Q1 대비 어떠했는지, 누적 수익률과 기여도 분석은 어떤지 질문)
response = await agent_q2.run(
    "How did CLIENT-001's Q2 performance compare to Q1? What was the cumulative return and attribution?",
    memory=memory_q2
)
print("Q2 성과 분석:")
print(response)
print("\n예상 결과: Q1 +8.2%, Q2 -2.1%, 누적 약 +5.9%, 방어적 포지셔닝이 도움됨")

## 7단계: Q3 자문 세션 - 회복 및 논제 업데이트

시장 회복으로 진행하고 투자 접근 방식을 업데이트합니다:

In [ ]:
# === Q3 자문 세션 ===
print("\n=== Q3: 시장 회복 및 논제 업데이트 ===")

agent_q3, memory_q3 = create_advisory_session("q3")

# 분기별 리뷰 미팅 기록
# (CLIENT-001의 분기별 리뷰 미팅 기록: 포트폴리오 가치 $3,450,000, 주요 결정사항 - 시장 회복 포지셔닝, 성장 배분 확대, 분산투자를 위한 국제 익스포저 추가)
response = await agent_q3.run(
    "Record client meeting for 'CLIENT-001' with meeting type 'Quarterly Review', "
    "portfolio value '$3,450,000', key decisions 'market recovery positioning, increase growth allocation, "
    "add international exposure for diversification'.",
    memory=memory_q3
)
print("Q3 분기별 리뷰:", response)

# 시장 진화에 기반한 투자 논제 업데이트
# (CLIENT-001의 국제 주식 투자 논제 업데이트: VTIAX를 통한 선진국 시장 익스포저 추가, 이머징 마켓 회복 잠재력, 중간 확신도)
response = await agent_q3.run(
    "Update investment thesis for 'CLIENT-001': asset class 'International Equity', "
    "thesis 'add developed market exposure via VTIAX, emerging markets recovery potential', conviction level 'medium'.",
    memory=memory_q3
)
print("Q3 국제 논제:", response)

In [ ]:
# 포괄적인 투자 이력 회상 테스트
# (CLIENT-001의 전체 투자 이력 요청: 모든 미팅, 성과 기간, 리밸런싱 조치, 투자 논제 진화 포함)
response = await agent_q3.run(
    "What is the complete investment history for CLIENT-001? Include all meetings, performance periods, "
    "rebalancing actions, and evolution of investment theses.",
    memory=memory_q3
)
print("Q3 전체 이력:")
print(response)
print("\n예상 결과: Q1 온보딩 → Q2 방어적 움직임 → Q3 회복 포지셔닝, 모든 성과 데이터")

## 8단계: Q4 자문 세션 - 연말 리뷰 및 계획

시맨틱 검색과 연간 성과 분석을 테스트합니다:

In [ ]:
# === Q4 자문 세션 ===
print("\n=== Q4: 연말 리뷰 및 계획 ===")

agent_q4, memory_q4 = create_advisory_session("q4")

# 연간 성과 추적
# (CLIENT-001의 포트폴리오 성과 추적: 2024 연간, 수익률 +12.8%, 벤치마크 수익률 +11.2%, 기여도 분석 - 전술적 포지셔닝 +1.1%, 섹터 배분 +0.5%)
response = await agent_q4.run(
    "Track portfolio performance for 'CLIENT-001': period '2024 Annual', return_pct '+12.8%', "
    "benchmark_return '+11.2%', attribution 'tactical positioning +1.1%, sector allocation +0.5%'.",
    memory=memory_q4
)
print("Q4 연간 성과:", response)

# 시장 인사이트 상관관계 테스트
# (올해 기록한 시장 인사이트는 무엇이었는지, CLIENT-001의 포트폴리오 결정에 어떤 영향을 미쳤는지 질문)
response = await agent_q4.run(
    "What market insights have I documented this year? How did they impact CLIENT-001's portfolio decisions?",
    memory=memory_q4
)
print("Q4 시장 인사이트 분석:")
print(response)
print("\n예상 결과: 지정학적 리스크 인사이트 → 방어적 포지셔닝 → 변동성 기간 초과 성과")

In [ ]:
# 유사한 포트폴리오 조치에 대한 시맨틱 검색 테스트
# (CLIENT-001에 대해 수행한 리밸런싱 조치는 무엇이었는지, 이후 성과 기준으로 가장 효과적이었던 것은 무엇인지 질문)
response = await agent_q4.run(
    "What rebalancing actions have I taken for CLIENT-001? Which were most effective based on subsequent performance?",
    memory=memory_q4
)
print("Q4 리밸런싱 분석:")
print(response)
print("\n예상 결과: Q2 방어적 조치 (QQQ 축소, VTV/VGSH 추가)가 변동성 기간에 도움됨")

## 9단계: 2년차 Q1 세션 - 다년 관점

장기 투자 지식과 고객 관계 진화를 테스트합니다:

In [ ]:
# === 2년차 Q1 자문 세션 ===
print("\n=== 2년차 Q1: 다년 관점 ===")

agent_y2q1, memory_y2q1 = create_advisory_session("year2-q1")

# 다년 포트폴리오 분석
# (CLIENT-001의 투자 여정 분석 요청: 지난 1년간 포트폴리오가 어떻게 진화했는지, 주요 결정과 그 결과는 무엇이었는지)
response = await agent_y2q1.run(
    "Analyze CLIENT-001's investment journey: How has their portfolio evolved over the past year? "
    "What were the key decisions and their outcomes?",
    memory=memory_y2q1
)
print("2년차 Q1 여정 분석:")
print(response)
print("\n예상 결과: $3.2M → $3.45M 성장, 방어적 포지셔닝 성공, 논제 진화")

In [ ]:
# 투자 논제 진화 추적 테스트
# (CLIENT-001에 대한 투자 논제가 어떻게 진화했는지, 어떤 자산 클래스를 추가했고 그 이유는 무엇인지 질문)
response = await agent_y2q1.run(
    "How have my investment theses for CLIENT-001 evolved? What asset classes have I added and why?",
    memory=memory_y2q1
)
print("2년차 Q1 논제 진화:")
print(response)
print("\n예상 결과: 미국 주식/채권으로 시작 → 국제 익스포저 추가 → 시장 상황에 따라 진화")

## 10단계: 최종 자산 관리 포트폴리오 평가

장기 투자 자문 역량에 대한 포괄적 테스트:

In [ ]:
# 최종 포괄적 자산 관리 포트폴리오 쿼리
# (전체 자산 관리 포트폴리오 제공 요청: 모든 고객의 투자 여정, 성과 기여도, 적용된 시장 인사이트, 리밸런싱 효과, 논제 진화 포함. 교훈과 개발된 베스트 프랙티스도 포함)
response = await agent_y2q1.run(
    "Provide my complete wealth management portfolio: all clients with their investment journeys, "
    "performance attribution, market insights applied, rebalancing effectiveness, and thesis evolution. "
    "Include lessons learned and best practices developed.",
    memory=memory_y2q1
)
print("전체 자산 관리 포트폴리오:")
print(response)
print("\n예상 결과: CLIENT-001의 전체 여정 - 성과 기여도, 시장 타이밍, 투자 진화 포함")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 이전에 논의된 정보를 회상할 수 있는지 확인"""
        # 실질적인 응답인지 확인 (단순히 "모르겠습니다"가 아닌지)
        has_content = len(response) > 50
        # 메모리 지표 확인
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        # 연결 언어 확인
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동하고 있습니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하지만, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 심각한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - 에이전트가 논의된 내용을 회상할 수 있는지 확인
# (이 세션에서 지금까지 무엇을 논의했는지 질문)
response1 = await agent_y2q1.run("What have we discussed so far in this session?", memory=memory_y2q1)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리 - 에이전트가 컨텍스트를 유지하는지 확인
# (이전에 무엇에 대해 이야기했는지 질문)
response2 = await agent_y2q1.run("What did we talk about earlier?", memory=memory_y2q1)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 능력 - 이전 컨텍스트와 연결할 수 있는지 확인
# (이것이 이전에 논의한 내용과 어떻게 관련되는지 질문)
response3 = await agent_y2q1.run("How does this relate to what we discussed before?", memory=memory_y2q1)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

**장기 메모리 통합**: 크로스 세션 자산 관리를 위한 AgentCore 메모리와 LlamaIndex 활용

**투자 여정 추적**: 여러 분기에 걸친 포트폴리오 진화 및 성과 기여도 분석

**시장 인텔리전스**: 시장 인사이트와 포트폴리오 적용의 시맨틱 검색

**투자 논제 진화**: 초기 포지셔닝에서 시장 적응형 전략으로의 자연스러운 발전

**성과 기여도**: 전술적 결정과 투자 결과에 대한 상세 추적

**자산 관리 우수성**: 시간에 걸친 포괄적 고객 관계 및 포트폴리오 최적화

투자 포트폴리오 어드바이저는 장기 메모리가 어떻게 어드바이저를 시간이 지남에 따라 더 현명해지는 지속적 자산 관리 파트너로 변환하는지 보여줍니다. 완전한 투자 이력을 유지하고 장기 고객 관계 전반에 걸쳐 정교한 금융 지식 검색을 가능하게 합니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다:

**참고**: 메모리를 영구적으로 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 노트북에서 앞서 생성한 메모리의 ID가 포함되어 있어야 합니다.

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    from bedrock_agentcore.memory import MemoryClient
    
    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
    
except NameError as e:
    print(f"경고: 변수가 정의되지 않았습니다: {e}")
    print("노트북을 처음부터 실행하거나 변수를 수동으로 설정하세요:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")